# 배치를 고려한 treated − control 비교

**질문:** 동일 배양 조건에서 24시간 처리의 효과를 배치와 분리해 추정할 수 있는가?

입력은 이 폴더의 `TASK.md`, `context.md`, `metadata.tsv`, `measurements.tsv`뿐이다. 제공 기록에 따르면 **합성 bulk marker 자료**이며, 값은 이미 정규화된 log2 intensity이다. 가상 marker 행과 sample ID 열로 구성되며, 사람 유래 배양 세포의 독립 donor 6명이 한 번씩 측정되었다. 기술 반복과 결측은 없다고 기록되어 있다. 본 분석은 합성 입력의 수치와 설계 검토이며 실제 생물학적 처리 효과의 검증이 아니다.

분석 방향은 treated − control, 기준은 control 및 batch A다. 쌍체 자료로 취급하지 않는다. 원자료의 추가 정규화, 로그 변환, 반올림, feature/sample 제외를 계획하지 않는다. 먼저 식별자·값·표본 관계를 확인하고, 실제 출력 확인 후 설계와 후속 분석을 결정한다.

**재실행:** 이 폴더에서 `python rerun_notebook.py`를 실행하거나 Jupyter에서 커널을 재시작하고 모든 셀을 순서대로 실행한다. Python, numpy, pandas, nbformat, nbclient, ipykernel이 필요하다. 버전과 입력 SHA-256은 아래 실행 출력에 기록된다. 무작위 계산은 사용하지 않는다.


## 1. 표본과 측정값을 그대로 비교할 수 있는가?

입력 기록을 출력하고, 중복 식별자·누락 표본·결측·비유한 값과 donor별 측정 수를 검사한다. 열 순서에 의존하지 않고 sample ID로 자료를 정렬한다. 여기서 통과하는 것은 자료의 구조적 점검이며 측정이나 생물학적 타당성 검증은 아니다.


In [1]:
from pathlib import Path
import csv
import hashlib
import importlib.metadata
import platform
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
input_paths = [root / name for name in ['TASK.md', 'context.md', 'metadata.tsv', 'measurements.tsv']]
print('분석 입력 디렉터리:', root)
print('제공 맥락:', (root / 'context.md').read_text().strip())
print('Python:', platform.python_version())
print('패키지:', {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'nbformat', 'nbclient', 'ipykernel']})
for path in input_paths:
    print(f'SHA256 {path.name}: {hashlib.sha256(path.read_bytes()).hexdigest()}')

with (root / 'measurements.tsv').open(newline='') as stream:
    raw_header = next(csv.reader(stream, delimiter='\t'))
metadata = pd.read_csv(root / 'metadata.tsv', sep='\t', dtype=str)
measurements = pd.read_csv(root / 'measurements.tsv', sep='\t')
required_columns = {'sample_id', 'donor_id', 'condition', 'batch'}
assert required_columns <= set(metadata.columns), 'Required metadata columns are absent'
assert measurements.columns[0] == 'feature_id', 'First measurement column must be feature_id'
assert len(raw_header) == len(set(raw_header)), 'Duplicate raw measurement header'
assert not metadata[list(required_columns)].isna().any().any(), 'Missing metadata value'
assert not metadata[list(required_columns)].eq('').any().any(), 'Empty metadata value'
assert metadata.sample_id.is_unique, 'Duplicate metadata sample_id'
assert measurements.feature_id.notna().all() and measurements.feature_id.is_unique, 'Invalid feature ID'
assert not measurements.feature_id.eq('').any(), 'Empty feature ID'
assert set(metadata.condition) == {'control', 'treated'}, 'Unexpected conditions'
assert set(metadata.batch) == {'A', 'B'}, 'Unexpected batches'
measurement_ids = list(measurements.columns[1:])
missing_in_measurements = sorted(set(metadata.sample_id) - set(measurement_ids))
missing_in_metadata = sorted(set(measurement_ids) - set(metadata.sample_id))
assert not missing_in_measurements and not missing_in_metadata, 'Sample identifier mismatch'
values = measurements.set_index('feature_id').loc[:, metadata.sample_id].apply(pd.to_numeric, errors='raise')
assert np.isfinite(values.to_numpy()).all(), 'Missing or nonfinite measurement'
assert metadata.donor_id.is_unique, 'Repeated donor: review experimental unit before proceeding'

audit = pd.Series({
    'samples_before': len(measurement_ids), 'samples_after': values.shape[1],
    'features_before': len(measurements), 'features_after': values.shape[0],
    'unique_donors': metadata.donor_id.nunique(),
    'duplicate_metadata_sample_ids': int(metadata.sample_id.duplicated().sum()),
    'duplicate_feature_ids': int(measurements.feature_id.duplicated().sum()),
    'missing_measurement_values': int(values.isna().sum().sum()),
    'sample_ids_missing_in_measurements': len(missing_in_measurements),
    'sample_ids_missing_in_metadata': len(missing_in_metadata),
    'minimum_log2_intensity': values.to_numpy().min(),
    'maximum_log2_intensity': values.to_numpy().max(),
}, name='observed')
display(audit.to_frame())
display(metadata)
display(values)
display(metadata.groupby('condition').agg(samples=('sample_id', 'size'), independent_donors=('donor_id', 'nunique')))


분석 입력 디렉터리: /tmp/bio-notebook-03-20261017
제공 맥락: 합성 bulk marker 자료. 값은 정규화된 log2 intensity, 행은 가상 marker, 열은 sample ID다. 사람 유래 배양 세포의 독립 donor 6명, 각 한 번 측정. control/treated는 동일 배양 조건에서 24시간 처리를 비교한다. 기술 반복과 결측은 없다.
Python: 3.12.7
패키지: {'numpy': '2.1.3', 'pandas': '2.3.1', 'nbformat': '5.10.4', 'nbclient': '0.10.2', 'ipykernel': '6.29.5'}
SHA256 TASK.md: 4df412fdb3691cb173cd50aaf0fc4070a6b8e9cd0d89b38128a2f7340c46dc19
SHA256 context.md: 540bce3b2ba0f37c8bab222e640e21d98098c70d7a6469ea887dcda58d20f4e4
SHA256 metadata.tsv: a3e6dd70a4f26218d386f7b909505eb22be0da922a1b53a942c1123698a0014f
SHA256 measurements.tsv: 939db602f9ef93849ea6c67da488e9883383ec6a5a7644a588fcc7bbc47a3385


,observed
samples_before,6.0
samples_after,6.0
features_before,2.0
features_after,2.0
unique_donors,6.0
duplicate_metadata_sample_ids,0.0
duplicate_feature_ids,0.0
missing_measurement_values,0.0
sample_ids_missing_in_measurements,0.0
sample_ids_missing_in_metadata,0.0


,sample_id,donor_id,condition,batch
0,C1,D1,control,A
1,C2,D2,control,A
2,C3,D3,control,A
3,T1,D4,treated,B
4,T2,D5,treated,B
5,T3,D6,treated,B


,C1,C2,C3,T1,T2,T3
feature_id,,,,,,
marker_A,5.0,5.2,4.8,7.0,7.2,6.8
marker_B,8.0,7.8,8.2,8.5,8.3,8.7


,samples,independent_donors
condition,,
control,3,3
treated,3,3


**실행 후 관찰:** 2개 marker와 6개 표본이 모두 매칭되며, 중복·결측·비유한 값은 없다. 원자료 전체를 유지했다(표본 6 → 6, marker 2 → 2). 값 범위는 4.8–8.7 log2 intensity이다. 각 군의 독립 표본은 donor 3명이며 donor가 군 간에 공유되지 않는다.

**해석과 다음 결정:** 정규화된 log2 intensity를 그대로 사용한다. raw count 모형이나 재로그 변환을 적용할 근거가 없다. 메타데이터에서 condition과 batch의 일치가 보였으므로, 처리 효과의 추정 가능성을 다음 단계에서 검증한다.


## 2. 처리와 배치의 효과를 분리할 수 있는가?

목표 모형은 `log2 intensity = 절편 + 처리 효과 × I(treated) + 배치 효과 × I(batch B) + 오차`이다. 기준은 control/batch A이며 처리 효과 방향은 treated − control이다. condition × batch 교차표와 설계행렬의 계수(rank)를 계산한다. rank는 서로 독립인 열의 수다. 세 계수를 분리하려면 세 열이 선형 독립이어야 한다. 아직 모형은 적합하지 않는다.


In [2]:
batch_table = pd.crosstab(metadata.condition, metadata.batch).reindex(index=['control', 'treated'], columns=['A', 'B'], fill_value=0)
display(batch_table)
design = pd.DataFrame({
    'intercept': np.ones(len(metadata), dtype=int),
    'treated_vs_control': metadata.condition.eq('treated').astype(int).to_numpy(),
    'batch_B_vs_A': metadata.batch.eq('B').astype(int).to_numpy(),
}, index=metadata.sample_id)
design.index.name = 'sample_id'
design_rank = np.linalg.matrix_rank(design.to_numpy())
display(design)
print(f'설계행렬: {design.shape[0]}행 × {design.shape[1]}열; rank = {design_rank}')
print('처리 열과 배치 열이 완전히 같은가:', design.treated_vs_control.equals(design.batch_B_vs_A))
null_direction = np.array([0.0, 1.0, -1.0])
print('X @ [0, +1, -1] =', (design.to_numpy() @ null_direction).tolist())
adjusted_effect_identifiable = design_rank == design.shape[1]
print('배치를 보정한 처리 효과 추정 가능:', adjusted_effect_identifiable)
if not adjusted_effect_identifiable:
    print('결정: batch-adjusted 처리 모형, 처리 효과 p값/CI 및 배치 보정값을 산출하지 않는다.')


batch,A,B
condition,,
control,3,0
treated,0,3


,intercept,treated_vs_control,batch_B_vs_A
sample_id,,,
C1,1,0,0
C2,1,0,0
C3,1,0,0
T1,1,1,1
T2,1,1,1
T3,1,1,1


설계행렬: 6행 × 3열; rank = 2
처리 열과 배치 열이 완전히 같은가: True
X @ [0, +1, -1] = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
배치를 보정한 처리 효과 추정 가능: False
결정: batch-adjusted 처리 모형, 처리 효과 p값/CI 및 배치 보정값을 산출하지 않는다.


**실행 후 관찰:** control은 batch A에만 3명, treated는 batch B에만 3명 있다. 목표 설계행렬은 6 × 3이지만 rank는 **2**다. 처리 지시변수와 배치 지시변수가 완전히 동일하며 `X @ [0, 1, -1] = 0`이다.

**해석:** 완전 교락이므로 관찰된 군 차이에서는 **처리 효과 + 배치 효과의 합**만 구분 가능하다. 처리 계수에 임의의 수를 더하고 배치 계수에서 같은 수를 빼도 모든 예측값이 동일하다. 따라서 배치를 고려한 처리 효과의 크기·부호·신뢰구간을 이 자료만으로 정할 수 없다. donor는 6명이어도 조건별 배치가 한 개씩이므로 donor 반복이 배치 교차를 대신하지 않는다.

**출력 확인 후 결정:** 배치 항을 제거한 결과나 의사역행렬의 특정 계수를 처리 효과로 보고하지 않는다. batch correction, 처리 검정, p값, 신뢰구간 및 FDR 보정은 수행하지 않는다. 검정 family를 구성하지 않았으므로 유의성에 대한 판단도 없다. 다음에는 각 군의 평균·표준편차·범위와 **treated/batch B − control/batch A의 관찰 차이**만 기술한다. 표준편차는 표본 간 퍼짐이며 처리 효과의 불확실성 구간이 아니다. 이 결정은 앞 단계의 rank 결과를 확인한 뒤 내렸다.


## 3. 실제 관찰된 군 차이는 얼마인가?

추가 변환 없이 원래 log2 intensity에서 군별 평균과 표본 표준편차(ddof=1), 최소·최댓값을 계산한다. 독립 donor 수를 함께 기록한다. 작은 표에 모든 관측값이 이미 제시되어 있으므로 그림은 추가하지 않는다. 관찰 차이는 처리와 배치가 합쳐진 비교이며 배치를 보정한 처리 효과가 아니다.


In [3]:
control_ids = metadata.loc[metadata.condition.eq('control'), 'sample_id'].tolist()
treated_ids = metadata.loc[metadata.condition.eq('treated'), 'sample_id'].tolist()
rows = []
for feature_id, row in values.iterrows():
    control = row.loc[control_ids]
    treated = row.loc[treated_ids]
    rows.append({
        'feature_id': feature_id,
        'n_control_donors': len(control), 'n_treated_donors': len(treated),
        'control_batch': 'A', 'treated_batch': 'B',
        'control_mean': control.mean(), 'treated_mean': treated.mean(),
        'control_sample_sd': control.std(ddof=1), 'treated_sample_sd': treated.std(ddof=1),
        'control_min': control.min(), 'control_max': control.max(),
        'treated_min': treated.min(), 'treated_max': treated.max(),
        'observed_treated_B_minus_control_A': treated.mean() - control.mean(),
        'units': 'normalized_log2_intensity',
        'batch_adjusted_treatment_effect': 'not_identifiable',
    })
descriptive_summary = pd.DataFrame(rows)
display(descriptive_summary)
print('군별 표본 표준편차와 범위:')
print(descriptive_summary[['feature_id', 'control_sample_sd', 'treated_sample_sd', 'control_min', 'control_max', 'treated_min', 'treated_max']].to_string(index=False))

# These are algebraic examples, not estimated treatment or batch effects.
illustrations = []
for row in descriptive_summary.itertuples(index=False):
    difference = row.observed_treated_B_minus_control_A
    for label, treatment, batch in [
        ('all_difference_assigned_to_treatment', difference, 0.0),
        ('all_difference_assigned_to_batch', 0.0, difference),
        ('opposite_treatment_direction_also_possible', -difference, 2.0 * difference),
    ]:
        beta = np.array([row.control_mean, treatment, batch])
        predicted = design.to_numpy() @ beta
        group_mean_predictions = np.where(metadata.condition.eq('treated'), row.treated_mean, row.control_mean)
        assert np.allclose(predicted, group_mean_predictions)
        illustrations.append({'feature_id': row.feature_id, 'algebraic_example_not_estimate': label,
                              'treatment_component': treatment, 'batch_component': batch,
                              'sum_matches_observed_difference': treatment + batch})
display(pd.DataFrame(illustrations))
descriptive_summary.to_csv(root / 'descriptive_group_differences.tsv', sep='\t', index=False, float_format='%.6g')
print('저장:', root / 'descriptive_group_differences.tsv')


,feature_id,n_control_donors,n_treated_donors,control_batch,treated_batch,control_mean,treated_mean,control_sample_sd,treated_sample_sd,control_min,control_max,treated_min,treated_max,observed_treated_B_minus_control_A,units,batch_adjusted_treatment_effect
0,marker_A,3,3,A,B,5.0,7.0,0.2,0.2,4.8,5.2,6.8,7.2,2.0,normalized_log2_intensity,not_identifiable
1,marker_B,3,3,A,B,8.0,8.5,0.2,0.2,7.8,8.2,8.3,8.7,0.5,normalized_log2_intensity,not_identifiable


군별 표본 표준편차와 범위:
feature_id  control_sample_sd  treated_sample_sd  control_min  control_max  treated_min  treated_max
  marker_A                0.2                0.2          4.8          5.2          6.8          7.2
  marker_B                0.2                0.2          7.8          8.2          8.3          8.7


,feature_id,algebraic_example_not_estimate,treatment_component,batch_component,sum_matches_observed_difference
0,marker_A,all_difference_assigned_to_treatment,2.0,0.0,2.0
1,marker_A,all_difference_assigned_to_batch,0.0,2.0,2.0
2,marker_A,opposite_treatment_direction_also_possible,-2.0,4.0,2.0
3,marker_B,all_difference_assigned_to_treatment,0.5,0.0,0.5
4,marker_B,all_difference_assigned_to_batch,0.0,0.5,0.5
5,marker_B,opposite_treatment_direction_also_possible,-0.5,1.0,0.5


저장: /tmp/bio-notebook-03-20261017/descriptive_group_differences.tsv


**실행 후 관찰:** marker_A의 평균은 control/A 5.0, treated/B 7.0으로 관찰 차이는 **+2.0 log2 intensity**다. marker_B는 8.0과 8.5로 차이가 **+0.5 log2 intensity**다. 모든 군·marker의 표본 표준편차는 0.2이며 각 군 n=3 donor다. 원래 sample별 값은 첫 번째 결과표에 모두 표시했다.

**해석:** 두 marker 모두 treated/B에서 관찰값이 높다. 그러나 marker_A의 (+처리 2, 배치 0), (처리 0, +배치 2), (처리 −2, +배치 4)는 모두 같은 군 평균을 설명한다. marker_B 역시 여러 분해가 가능하다. 위 표의 성분값은 추정 결과나 현실적 배치 크기의 주장 없이, 식별 불가능성을 보여주는 대수 예시다. 따라서 "처리가 증가시켰다" 또는 "처리 효과가 없다"는 결론을 낼 수 없다.

## 현재 답과 다음 확인 사항

**현재 답:** 배치를 고려한 24시간 처리 효과는 이 자료에서 추정할 수 없다. 실행 가능한 자료 무결성 점검, 표본 수 확인, 설계행렬 rank 점검, 가상 marker 2개의 기술통계 및 비식별성 확인을 완료했다. 원자료는 변경하지 않았다. 그림은 만들지 않았고 시각 검토 대상도 없다.

확인된 사실은 제공 기록과 파일에서 구분한다. 독립 donor·측정 단위·합성 자료라는 점은 `context.md`에 근거하며, donor ID의 유일성·표본 수·완전 교락은 파일에서 직접 확인했다. 원 실험 기록이나 생물학적 독립성을 외부에서 검증한 것은 아니다. 정규화 방법, 배치가 배양·처리·측정 중 어느 단계인지, 배정 무작위화 여부, 합성 데이터 생성 규칙은 제공되지 않았다.

다음 조치는 우선순위 순이다.

1. **sample–condition–batch 매핑과 배치의 의미를 원기록으로 확인한다.** 기록 오류가 있다면 원본을 보존하고 확인된 수정본으로 설계와 결과를 재계산한다. 분석을 가능하게 만들 목적으로 라벨을 바꾸지 않는다.
2. **각 배치에 두 조건이 모두 들어가도록 새로운 비교 자료를 확보한다.** 예를 들어 여러 배치 각각에서 control과 treated를 함께 배양·처리·측정하고 독립 donor를 배치에 균형 있게 배정한다. 기존처럼 control/A와 treated/B만 늘려서는 교락이 해소되지 않는다. 필요한 donor 수는 과학적으로 의미 있는 차이와 별도로 확보한 변동 정보에 따라 정하며, 현재 합성 표의 SD=0.2를 실제 연구의 검정력 근거로 사용하지 않는다.
3. **동일 donor의 세포를 두 조건에 나눌 수 있다면 donor 내 비교를 설계한다.** 이 경우 donor가 독립 단위이고 두 조건 측정값은 쌍체이므로 donor를 반영한 분석으로 바꾼다. 현재의 서로 다른 donor를 임의로 짝짓지는 않는다.
4. **이미 측정한 동일 물질을 배치 간 재측정할 수 있는지 확인한다.** 공통 reference/bridge sample은 측정 단계의 배치 차이 평가에 도움이 될 수 있으나, 배양·처리 단계의 교락이나 donor 차이를 자동으로 해결하지 않는다. 기술 반복을 새로운 독립 donor로 세지 않는다.

교차 설계와 입력 의미가 확보된 뒤에야 `condition + batch` 및 필요한 donor 구조를 반영한 모형의 추정 가능성을 다시 확인하고, 처리 효과·불확실성·다중검정을 계획한다. 현 자료에서 처리 검정, 보정된 효과/CI, 배치 교정, 기전 해석 및 실제 생물학적 검증은 수행하지 않았다.

**산출물:** 이 노트북(질문·코드·실제 출력·후속 결정), `descriptive_group_differences.tsv`(재사용 가능한 기술통계), `rerun_notebook.py`(새 커널 전체 재실행). 노트북의 환경 버전과 입력 SHA-256은 첫 실행 셀 출력에 남는다.


<!-- rerun_status -->
**전체 재실행 확인:** 2026-10-17T08:28:13+00:00에 새 Python 커널에서 코드 셀 3개를 순서대로 실행했고 오류 없이 실제 출력을 저장했다. 원래 입력 4개 파일의 실행 전후 SHA-256은 동일했다. 노트북 스키마 검사도 통과했다. 최초 탐색은 실제 노트북 커널에서 단계별로 실행했고, 각 단계 출력을 확인한 뒤 다음 단계와 해석을 추가했다. 이 재실행은 코드와 저장 출력의 일치를 확인하며 생물학적 검증은 아니다. 그림을 생성하지 않아 그림 시각 검토는 해당 없음.